# 11.4 Serving Many Clients

**Prerequisites:** 11.2 TCP Client and Server, 04 Functions (closures), 06 Exception Handling  
**Target:** Python 3.12+ (notes flag 3.13/3.14 differences)

### What you'll learn
- 🔴 Why a straightforward server serves exactly **one** client at a time
- Thread per client - the simple fix, and what it costs
- `ThreadPoolExecutor` - bounding the damage
- **Non-blocking I/O** and `selectors`: thousands of clients, one thread
- `asyncio` streams - the same idea with readable code
- 🔴 Running `asyncio` inside Jupyter, where a loop is already going
- Choosing between threads, `selectors` and `asyncio`

---

## 🔴 The problem

The server in **11.2** looked fine. It has a fatal flaw:

```
    while True:
        conn, addr = srv.accept()      <- waits here
        handle(conn)                   <- ...and here, for as long as this
                                          client takes. Nobody else is served.
```

One slow client blocks **every** other client. If `handle()` takes two seconds, ten clients take twenty seconds, and the tenth waits eighteen of them.

This is not a theoretical concern — it is the normal state of a first server. The cell below measures it.

In [ ]:
import socket
import threading
import time

WORK_TIME = 0.3          # pretend each request does something slow
CLIENTS = 6


def serial_server(stop_flag, address_box):
    """One client at a time - the naive loop."""
    srv = socket.socket()
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("127.0.0.1", 0))
    srv.listen(16)                    # backlog: they queue here, unserved
    srv.settimeout(0.2)
    address_box.append(srv.getsockname())
    while not stop_flag.is_set():
        try:
            conn, _ = srv.accept()
        except (TimeoutError, socket.timeout):
            continue
        with conn:
            conn.settimeout(5.0)
            conn.recv(1024)
            time.sleep(WORK_TIME)     # the slow bit
            conn.sendall(b"done")
    srv.close()


def hit(address, results, index):
    started = time.perf_counter()
    with socket.create_connection(address, timeout=20.0) as sock:
        sock.sendall(b"work")
        sock.recv(1024)
    results[index] = time.perf_counter() - started


def measure(server_target, label):
    stop_flag, box = threading.Event(), []
    server_thread = threading.Thread(
        target=server_target, args=(stop_flag, box), daemon=True)
    server_thread.start()
    while not box:
        time.sleep(0.01)

    results = [0.0] * CLIENTS
    started = time.perf_counter()
    threads = [threading.Thread(target=hit, args=(box[0], results, i))
               for i in range(CLIENTS)]
    for t in threads:
        t.start()
    for t in threads:
        t.join(timeout=30)
    total = time.perf_counter() - started

    stop_flag.set()
    server_thread.join(timeout=5)
    print(f"{label}")
    print(f"  {CLIENTS} clients, each needing {WORK_TIME}s of server work")
    print(f"  slowest client waited : {max(results):.2f}s")
    print(f"  total wall clock      : {total:.2f}s")
    return total


serial_total = measure(serial_server, "SERIAL SERVER")
print(f"\n  ^ about {CLIENTS} x {WORK_TIME} = {CLIENTS * WORK_TIME:.1f}s.")
print("    The clients were served strictly one after another.")

## Fix 1: a thread per client

Accept the connection, hand it to a thread, go straight back to accepting. The `accept()` loop stays free, so clients are served concurrently.

This works, and it is the right answer more often than people admit — a server handling a few hundred concurrent connections is entirely fine on threads.

### What it costs

| | |
|---|---|
| Memory | each thread has its own stack — typically ~8 MB of address space |
| Context switching | the OS scheduler does more work as thread count climbs |
| Practical ceiling | hundreds to a few thousand; not 100,000 |

> **What about the GIL?** It does not hurt here. A thread waiting on `recv()` **releases the GIL**, so blocked threads cost nothing in contention. The GIL limits CPU-bound parallelism, not I/O concurrency — which is exactly why threads suit network servers. See **12** for the full story.

In [ ]:
def threaded_server(stop_flag, address_box):
    """Hand each connection to its own thread."""
    srv = socket.socket()
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("127.0.0.1", 0))
    srv.listen(16)
    srv.settimeout(0.2)
    address_box.append(srv.getsockname())
    workers = []

    def handle(conn):
        with conn:
            conn.settimeout(5.0)
            conn.recv(1024)
            time.sleep(WORK_TIME)
            conn.sendall(b"done")

    while not stop_flag.is_set():
        try:
            conn, _ = srv.accept()
        except (TimeoutError, socket.timeout):
            continue
        worker = threading.Thread(target=handle, args=(conn,), daemon=True)
        worker.start()
        workers.append(worker)
        # accept() comes round again immediately - that is the whole trick

    for worker in workers:
        worker.join(timeout=5)
    srv.close()


threaded_total = measure(threaded_server, "THREAD PER CLIENT")
print(f"\n  serial: {serial_total:.2f}s -> threaded: {threaded_total:.2f}s")
print(f"  {serial_total / threaded_total:.1f}x faster, and the slowest client now")
print(f"  waits about {WORK_TIME}s instead of queueing behind everyone else.")

### Bounding it: `ThreadPoolExecutor`

An unbounded thread-per-client server has an obvious attack: open 50,000 connections and the machine falls over. A **pool** caps the number of workers — beyond that, clients queue.

That is a deliberate trade: bounded resource use, at the cost of latency under load. Which is almost always the right choice for a real service, because falling over is worse than being slow.

In [ ]:
from concurrent.futures import ThreadPoolExecutor

MAX_WORKERS = 3          # deliberately fewer than CLIENTS, to show queueing


def pooled_server(stop_flag, address_box):
    srv = socket.socket()
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("127.0.0.1", 0))
    srv.listen(16)
    srv.settimeout(0.2)
    address_box.append(srv.getsockname())

    def handle(conn):
        with conn:
            conn.settimeout(5.0)
            conn.recv(1024)
            time.sleep(WORK_TIME)
            conn.sendall(b"done")

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        while not stop_flag.is_set():
            try:
                conn, _ = srv.accept()
            except (TimeoutError, socket.timeout):
                continue
            pool.submit(handle, conn)
    srv.close()


pooled_total = measure(pooled_server, f"THREAD POOL (max {MAX_WORKERS})")
expected = (CLIENTS / MAX_WORKERS) * WORK_TIME
print(f"\n  {CLIENTS} clients through {MAX_WORKERS} workers ~= {expected:.1f}s")
print("  Slower than unbounded threads - and it cannot be swamped.")

---

## Fix 2: non-blocking I/O with `selectors`

Threads spend their lives *waiting*. The alternative is to stop waiting: set every socket **non-blocking**, then ask the OS "which of these is ready right now?" and only touch those.

That question is what `select`/`poll`/`epoll`/`kqueue` answer, and **`selectors`** is Python's portable wrapper — it picks the best mechanism your platform offers.

```
    sel.register(sock, EVENT_READ, data=...)   tell it what to watch
    events = sel.select(timeout=...)           block until SOMETHING is ready
    for key, mask in events:                   handle only those
```

One thread, one loop, thousands of connections. This *is* an event loop — you are writing by hand what `asyncio` does for you.

🔴 The cost is that **you may never block**. Every handler must do a little work and return. A single `time.sleep()` or slow database call in the loop stalls every connection — the same flaw as the serial server, in a new disguise.

In [ ]:
import selectors
import types

print("selector this platform gives us:", selectors.DefaultSelector.__name__)


def selector_server(stop_flag, address_box):
    """One thread, many clients, nothing blocking."""
    sel = selectors.DefaultSelector()
    srv = socket.socket()
    srv.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    srv.bind(("127.0.0.1", 0))
    srv.listen(16)
    srv.setblocking(False)
    address_box.append(srv.getsockname())
    sel.register(srv, selectors.EVENT_READ, data=None)

    while not stop_flag.is_set():
        for key, _mask in sel.select(timeout=0.2):
            if key.data is None:
                # the listening socket is ready -> a client is waiting
                conn, _addr = key.fileobj.accept()
                conn.setblocking(False)
                sel.register(
                    conn, selectors.EVENT_READ,
                    data=types.SimpleNamespace(outgoing=b""),
                )
            else:
                conn = key.fileobj
                try:
                    received = conn.recv(1024)
                except BlockingIOError:
                    continue                      # not actually ready
                if received:
                    # NOTE: no sleep here - blocking would stall everyone
                    conn.sendall(b"done")
                sel.unregister(conn)
                conn.close()

    sel.close()
    srv.close()


stop_flag, box = threading.Event(), []
thread = threading.Thread(target=selector_server, args=(stop_flag, box), daemon=True)
thread.start()
while not box:
    time.sleep(0.01)

MANY = 40
results = [0.0] * MANY
started = time.perf_counter()
clients = [threading.Thread(target=hit, args=(box[0], results, i)) for i in range(MANY)]
for t in clients:
    t.start()
for t in clients:
    t.join(timeout=30)
elapsed = time.perf_counter() - started

stop_flag.set()
thread.join(timeout=5)
print(f"\n{MANY} clients served by ONE thread in {elapsed:.2f}s")
print(f"slowest client waited {max(results):.3f}s")
print("\nNo per-client threads at all. This scales to thousands of")
print("connections - which is why every high-performance server works this way.")

---

## Fix 3: `asyncio`

`selectors` works, and the code is awkward: state machines, manual buffers, no obvious flow. `asyncio` runs the same event loop but lets you write each connection as if it were sequential.

```
    async def handle(reader, writer):      one coroutine per connection,
        data = await reader.readline()     reading top to bottom...
        writer.write(b"done")
        await writer.drain()               ...but `await` yields control,
                                           so other connections run meanwhile
```

`await` is the key: it marks the points where this connection is waiting and something else may run. One thread, thousands of connections, and code that reads sequentially.

`asyncio.start_server` also gives you `reader.readline()` — the newline framing from **11.2**, already handled.

### 🔴 `asyncio.run()` inside Jupyter

Jupyter **already runs an event loop** to talk to the notebook front end. So:

```
    asyncio.run(main())     -> RuntimeError: asyncio.run() cannot be called
                               from a running event loop
```

There are two answers, and which you need depends on where the code runs:

| Where | What works |
|---|---|
| A `.py` script | `asyncio.run(main())` |
| A Jupyter cell | bare `await main()` — Jupyter allows top-level `await` |
| **Both** | the `run_async` helper below |

The helper checks whether a loop is already running and, if so, runs the coroutine on a separate thread with its own loop. That keeps this notebook working under Jupyter *and* under a plain interpreter.

In [ ]:
import asyncio
from concurrent.futures import ThreadPoolExecutor as _Pool


def run_async(coro):
    """Run a coroutine whether or not an event loop is already going."""
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)          # no loop: a plain script
    # A loop IS running (Jupyter). asyncio.run() would raise, so give the
    # coroutine its own loop on its own thread.
    with _Pool(1) as pool:
        return pool.submit(asyncio.run, coro).result()


async def handle_client(reader, writer):
    request = await reader.readline()          # newline framing, for free
    await asyncio.sleep(WORK_TIME)             # await, NOT time.sleep
    writer.write(b"done:" + request)
    await writer.drain()
    writer.close()
    await writer.wait_closed()


async def async_demo(n_clients):
    server = await asyncio.start_server(handle_client, "127.0.0.1", 0)
    port = server.sockets[0].getsockname()[1]

    async def one_client(i):
        reader, writer = await asyncio.open_connection("127.0.0.1", port)
        writer.write(f"job-{i}\n".encode())
        await writer.drain()
        reply = await asyncio.wait_for(reader.readline(), timeout=10)
        writer.close()
        await writer.wait_closed()
        return reply.strip()

    started = asyncio.get_running_loop().time()
    replies = await asyncio.gather(*(one_client(i) for i in range(n_clients)))
    elapsed = asyncio.get_running_loop().time() - started

    server.close()
    await server.wait_closed()
    return replies, elapsed


replies, elapsed = run_async(async_demo(CLIENTS))
print(f"{CLIENTS} clients, each needing {WORK_TIME}s of work")
print(f"total: {elapsed:.2f}s  (serial would be {CLIENTS * WORK_TIME:.1f}s)")
print("first three replies:", replies[:3])
print("\nOne thread. No locks. Code that reads top to bottom.")

### 🔴 One blocking call ruins it

`asyncio` is cooperative: a coroutine keeps the loop until it hits an `await`. Call something that blocks — `time.sleep()`, `requests.get()`, a synchronous database driver — and **every** connection stops, because the single thread is stuck.

The next cell shows the identical server with `time.sleep()` instead of `asyncio.sleep()`. Same code shape, concurrency gone.

The fix when you must call blocking code is `asyncio.to_thread()`, which moves it to a worker thread and awaits the result.

In [ ]:
async def blocking_handler(reader, writer):
    request = await reader.readline()
    time.sleep(WORK_TIME)                  # 🔴 BLOCKS THE ENTIRE LOOP
    writer.write(b"done:" + request)
    await writer.drain()
    writer.close()
    await writer.wait_closed()


async def blocking_demo(n_clients):
    server = await asyncio.start_server(blocking_handler, "127.0.0.1", 0)
    port = server.sockets[0].getsockname()[1]

    async def one_client(i):
        reader, writer = await asyncio.open_connection("127.0.0.1", port)
        writer.write(f"job-{i}\n".encode())
        await writer.drain()
        await asyncio.wait_for(reader.readline(), timeout=30)
        writer.close()
        await writer.wait_closed()

    started = asyncio.get_running_loop().time()
    await asyncio.gather(*(one_client(i) for i in range(n_clients)))
    elapsed = asyncio.get_running_loop().time() - started
    server.close()
    await server.wait_closed()
    return elapsed


blocked = run_async(blocking_demo(CLIENTS))
print(f"with asyncio.sleep : {elapsed:.2f}s   <- concurrent")
print(f"with time.sleep    : {blocked:.2f}s   <- serial again")
print(f"\n{blocked / elapsed:.1f}x slower, from changing one line.")
print("\nNothing warns you. The code looks asynchronous and is not -")
print("which is why 'is this library async-safe?' matters so much.")

## Choosing

| | Threads | `selectors` | `asyncio` |
|---|---|---|---|
| Concurrent connections | hundreds | thousands | thousands |
| Memory per connection | ~8 MB stack | a few KB | a few KB |
| Code style | sequential, natural | state machine, awkward | sequential, natural |
| Blocking library calls | fine | 🔴 breaks it | 🔴 breaks it (use `to_thread`) |
| Shared state | needs locks | single-threaded | single-threaded |
| Debugging | familiar | fiddly | improving, still harder |

### A straight recommendation

- **A few hundred connections, and existing blocking libraries?** Threads. Simplest thing that works, and "it does not scale to 100k" is irrelevant if you will never have 100k.
- **Many connections, mostly waiting on I/O?** `asyncio`, provided your libraries have async versions.
- **`selectors` directly?** Rarely — when you need precise control, or to understand what `asyncio` is doing.

> **Do not skip the boring option.** A thread-per-client server behind a load balancer handles more traffic than most applications ever see, and every developer can read it.

In [ ]:
# ---- confirm nothing was left running ----
alive = [t.name for t in threading.enumerate() if t is not threading.main_thread()]
print("background threads still alive:", alive or "none")
print("\nEvery server here was stopped explicitly, and all ran as daemon")
print("threads - so none could outlive the interpreter even if it were not.")

---

## Common Mistakes & Pitfalls

1. 🔴 **Calling the handler directly in the `accept()` loop.** That is a serial server; one slow client blocks everyone.
2. 🔴 **Blocking inside an `asyncio` coroutine or a `selectors` loop.** `time.sleep`, `requests`, or a synchronous DB driver stops every connection. Use `asyncio.to_thread()`.
3. 🔴 **Calling `asyncio.run()` in a Jupyter cell.** A loop is already running. Use bare `await`, or a helper like `run_async`.
4. **Unbounded thread-per-client.** 50,000 connections becomes 50,000 threads. Use a pool.
5. **Blaming the GIL for slow I/O servers.** Threads waiting on `recv()` release the GIL; it limits CPU parallelism, not I/O concurrency.
6. **Forgetting `setblocking(False)` on sockets accepted in a `selectors` loop.** The listening socket being non-blocking does not make its children so.
7. **Sharing mutable state across handler threads without a lock.** See **12**.
8. **Reaching for `asyncio` because it is modern.** If your database driver is synchronous, you get the complexity and none of the benefit.

## Best Practices

- Start with threads. Move to `asyncio` when you have measured that you need to.
- Bound your concurrency - a thread pool, or a semaphore around async tasks.
- In async code, check every library is non-blocking; wrap the ones that are not in `asyncio.to_thread()`.
- Set timeouts on both sides; `asyncio.wait_for()` is the async equivalent.
- Keep per-connection state in an object attached to the connection, not in globals.
- Make handlers exception-safe - one client's error must not kill the server loop.
- Prefer `asyncio.start_server` over hand-rolled `selectors` unless you have a reason.
- Test with more concurrent clients than you expect. Serial behaviour hides at low load.

## Practice Exercises

Try these before moving on.

1. Raise `CLIENTS` to 50 and re-run the serial and threaded servers. Does the ratio hold? Where does thread-per-client start to degrade on your machine?
2. Add a `Semaphore` to the asyncio demo limiting in-flight requests to 3, and confirm the timing matches the thread-pool version.
3. Replace `time.sleep(WORK_TIME)` in `blocking_handler` with `await asyncio.to_thread(time.sleep, WORK_TIME)` and show concurrency returns.
4. Extend the `selectors` server to echo the request back, which needs buffering the outgoing data and registering `EVENT_WRITE`. Note how much harder it just got.
5. Make the threaded server survive a handler that raises. Which clients are affected, and which are not?
6. 🔴 Measure memory: start 500 idle connections against the threaded server and then the asyncio server, comparing process memory. Explain the difference.
7. Write a chat server where a message from any client is broadcast to all others - once with threads and a lock, once with asyncio and no lock. Which was easier to get right?